# ASSIGNMENT: Cartoon Face Mask
## This assignment tests your ability to apply face recognition, image masks, image thresholding, video capturing and saving video feed into a video file concepts.

## TASK: Cartoonify faces in video feed from live webcam

### Steps
- 1. **Capture video** feed from webcam
- 2. **Recognize faces** in the video
- 3. **Replace/Mask the face** region with your favorite cartoon character
- 4. **Save the video** feed into a video file
- 5. Submit the following files
    - Notebook (.ipynb)
    - Video (.avi/.mp4)

### Helper code to recognize faces

In [1]:
import cv2
import os

# 載入臉部辨識的分類器
cascPathface = os.path.dirname(cv2.__file__) + "/data/haarcascade_frontalface_alt2.xml"
faceCascade = cv2.CascadeClassifier(cascPathface)

# 載入卡通圖像，這裡使用透明背景 PNG
cartoon_img = cv2.imread("Cartoon mask.png", cv2.IMREAD_UNCHANGED)  # 請將 cartoon.png 換成你想要的圖片名稱

# 啟動攝影機
video_capture = cv2.VideoCapture(0)

# 取得畫面尺寸資訊（用於儲存影片）
frame_width = int(video_capture.get(3))
frame_height = int(video_capture.get(4))

# 初始化影片輸出（儲存為 AVI）
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('Video.mp4', fourcc, 20.0, (frame_width, frame_height))

# 主迴圈
while True:
    ret, frame = video_capture.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = faceCascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

    for (x, y, w, h) in faces:
        # 調整卡通圖片大小與臉部區域一致
        resized_cartoon = cv2.resize(cartoon_img, (w, h))

        # 判斷是否包含透明度 alpha channel
        if resized_cartoon.shape[2] == 4:
            # 分離 BGR 與 Alpha mask
            b, g, r, a = cv2.split(resized_cartoon)
            mask = cv2.merge((a, a, a))
            cartoon_rgb = cv2.merge((b, g, r))

            # 將原始畫面中的臉部區域與卡通圖疊合
            roi = frame[y:y+h, x:x+w]
            roi_bg = cv2.bitwise_and(roi, cv2.bitwise_not(mask))
            roi_fg = cv2.bitwise_and(cartoon_rgb, mask)
            dst = cv2.add(roi_bg, roi_fg)
            frame[y:y+h, x:x+w] = dst
        else:
            frame[y:y+h, x:x+w] = resized_cartoon

    # 顯示畫面
    cv2.imshow('Cartoon Face Video', frame)

    # 寫入影片
    out.write(frame)

    # 按下 'q' 鍵離開
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 釋放資源
video_capture.release()
out.release()
cv2.destroyAllWindows()
